In [2]:
# Delete everything inside /kaggle/working
!rm -rf /kaggle/working/*

In [3]:
import os
import torch
import random
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)

# ── 1. Config (mirrors the real prepare_ADA_splits) ──────────────────────────
BASE = Path("/kaggle/working")

dataset_roots = {
    "CodecFake":    BASE / "audio/CodecFake/fake",
    "ASVspoof2021": BASE / "audio/ASVspoof2021/fake",
    "FakeOrReal":   BASE / "audio/FOR/fake"
}
dataset_labels = {
    "CodecFake": 0,
    "ASVspoof2021": 1,
    "FakeOrReal": 2
}

N_PER_DATASET   = 50       # small enough to be instant
SAMPLE_RATE     = 16000
DURATION_S      = 2
N_SAMPLES       = SAMPLE_RATE * DURATION_S   # [1, 32000] tensors

# ── 2. Generate .pt files ─────────────────────────────────────────────────────
for name, root in dataset_roots.items():
    root.mkdir(parents=True, exist_ok=True)
    
    if name == "CodecFake":
        # Distribute evenly across F01-F06
        for i in range(N_PER_DATASET):
            cls = (i % 6) + 1          # cycles 1→2→3→4→5→6→1→...
            tensor = torch.randn(1, N_SAMPLES).float()
            torch.save(tensor, root / f"F0{cls}_sample_{i:04d}.pt")
    else:
        for i in range(N_PER_DATASET):
            tensor = torch.randn(1, N_SAMPLES).float()
            torch.save(tensor, root / f"sample_{i:04d}.pt")
    
    print(f"Generated {N_PER_DATASET} .pt files → {root}")

Generated 50 .pt files → /kaggle/working/audio/CodecFake/fake
Generated 50 .pt files → /kaggle/working/audio/ASVspoof2021/fake
Generated 50 .pt files → /kaggle/working/audio/FOR/fake


# AUTOENCODER CODE

## Autoencoder Training

In [4]:
import re
import torch
import random
from pathlib import Path
from torch.utils.data import Dataset

class CodecFakeMultiClassDataset(Dataset):
    def __init__(self, root_dir, seed=None):
        self.samples = []
        root = Path(root_dir)

        for file_path in sorted(root.glob("*.pt")):
            match = re.search(r"F(\d+)_", file_path.name)
            if match:
                label = int(match.group(1))
                self.samples.append((file_path, label))

        # Print the number of samples for each class
        class_counts = {}
        for _, label in self.samples:
            if label in class_counts:
                class_counts[label] += 1
            else:
                class_counts[label] = 1
        for label, count in class_counts.items():
            print(f"Class {label}: {count} samples")
        # Print the total number of samples
        print(f"Total samples: {len(self.samples)}")

        # Shuffle the samples
        if seed is not None:
            rng = random.Random(seed)
            rng.shuffle(self.samples)
        else:
            random.shuffle(self.samples)


        if not self.samples:
            raise ValueError("No valid .pt files found or unable to extract labels.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        tensor = torch.load(path).float()
        label = label - 1  # Convert to zero-based index for classification tasks
        return tensor, label


In [5]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
from tqdm import tqdm
from torch.utils.data import DataLoader

In [6]:
class DeepAutoencoder(nn.Module):
    def __init__(self):
        super(DeepAutoencoder, self).__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=9, stride=2, padding=4),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Conv1d(32, 64, kernel_size=9, stride=2, padding=4),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Conv1d(64, 128, kernel_size=9, stride=2, padding=4),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Conv1d(128, 256, kernel_size=9, stride=2, padding=4),
            nn.BatchNorm1d(256),
            nn.ReLU(),
        )

        # Decoder with dropout
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(256, 128, kernel_size=9, stride=2, padding=4, output_padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.ConvTranspose1d(128, 64, kernel_size=9, stride=2, padding=4, output_padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.ConvTranspose1d(64, 32, kernel_size=9, stride=2, padding=4, output_padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Dropout(0.3),
            nn.ConvTranspose1d(32, 1, kernel_size=9, stride=2, padding=4, output_padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out

In [7]:
def train(model, train_loader, val_loader, device, epochs=50, lr=1e-4, save_path='/kaggle/working/models/autoencoder.pt'):
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.SmoothL1Loss(beta=0.0001)
    best_val_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for x, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            x = x.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, x)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, _ in val_loader:
                x = x.to(device)
                out = model(x)
                loss = criterion(out, x)
                val_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        print(f"Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            if os.path.exists(save_path):
                os.remove(save_path)
            torch.save(model.state_dict(), save_path)
            print(f"Model saved to {save_path}")

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_dir = "/kaggle/working"

dataset = CodecFakeMultiClassDataset(
    root_dir="/kaggle/working/audio/CodecFake/fake"
)

model_dir = Path("/kaggle/working/models")
model_dir.mkdir(parents=True, exist_ok=True)

# Split dataset into training and validation sets
total_len = len(dataset)
train_len = int(0.8 * total_len)
val_len = total_len - train_len
seed = 42  # Set a seed for reproducibility
generator = torch.Generator().manual_seed(seed)
train_set, val_set = data.random_split(dataset, [train_len, val_len], generator=generator)

print(f"Total samples: {total_len}")
print(f"Training samples: {len(train_set)}")
print(f"Validation samples: {len(val_set)}")

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader = DataLoader(val_set, batch_size=16)

model = DeepAutoencoder().to(device)
save_path = '/kaggle/working/models/autoencoder.pt'
print(f"Model will be saved to {save_path}")
train(model, train_loader, val_loader, device, epochs=50, lr=1e-4, save_path=save_path)

Class 1: 9 samples
Class 2: 9 samples
Class 3: 8 samples
Class 4: 8 samples
Class 5: 8 samples
Class 6: 8 samples
Total samples: 50
Total samples: 50
Training samples: 40
Validation samples: 10
Model will be saved to /kaggle/working/models/autoencoder.pt


Epoch 1/50: 100%|██████████| 3/3 [00:02<00:00,  1.37it/s]


Epoch 01 | Train Loss: 0.9937 | Val Loss: 0.8297
Model saved to /kaggle/working/models/autoencoder.pt


Epoch 2/50: 100%|██████████| 3/3 [00:00<00:00, 13.52it/s]


Epoch 02 | Train Loss: 0.9296 | Val Loss: 0.8631


...

Epoch 48 | Train Loss: 0.4459 | Val Loss: 0.3696
Model saved to /kaggle/working/models/autoencoder.pt


Epoch 49/50: 100%|██████████| 3/3 [00:00<00:00, 14.37it/s]


Epoch 49 | Train Loss: 0.4438 | Val Loss: 0.3675
Model saved to /kaggle/working/models/autoencoder.pt


Epoch 50/50: 100%|██████████| 3/3 [00:00<00:00, 14.51it/s]

Epoch 50 | Train Loss: 0.4420 | Val Loss: 0.3658
Model saved to /kaggle/working/models/autoencoder.pt


# Audio Deepfake Attribution (ADA)

## Prepare ADA Data Splits

In [10]:
import os
import random
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)

def prepare_ADA_splits():
    
    # paths
    dataset_roots = {
        "CodecFake": "/kaggle/working/audio/CodecFake/fake",
        "ASVspoof2021": "/kaggle/working/audio/ASVspoof2021/fake",
        "FakeOrReal": "/kaggle/working/audio/FOR/fake"
    }

    # Labels
    dataset_labels = {
        "CodecFake": 0,
        "ASVspoof2021": 1,
        "FakeOrReal": 2
    }

    # Quotas to use for each dataset
    MAX_SAMPLES_PER_DATASET = 25000

    # Output
    output_csv_dir = "/kaggle/working/csv/ADA_split"
    os.makedirs(output_csv_dir, exist_ok=True)

    # Collect samples from all datasets
    all_samples = []

    for name, root in dataset_roots.items():
        files = list(Path(root).rglob("*.pt"))
        files = sorted(files)  # deterministic order

        if len(files) < MAX_SAMPLES_PER_DATASET:
            print(f"Warning: {name} has only {len(files)} files. Using all.")
            selected = files
        else:
            selected = random.sample(files, MAX_SAMPLES_PER_DATASET)

        label = dataset_labels[name]
        all_samples.extend([(str(f.resolve()), label) for f in selected])
        print(f"{name}: selected {len(selected)} samples (label={label})")


    # Shuffle and split
    random.shuffle(all_samples)

    train_val, test = train_test_split(all_samples, test_size=0.2, stratify=[l for _, l in all_samples], random_state=SEED)
    train, val = train_test_split(train_val, test_size=0.25, stratify=[l for _, l in train_val], random_state=SEED)
    # Result: 60% train, 20% val, 20% test

    splits = {"train": train, "val": val, "test": test}

    # Saving CSV
    for name, split in splits.items():
        df = pd.DataFrame(split, columns=["path", "label"])
        csv_path = os.path.join(output_csv_dir, f"{name}.csv")
        df.to_csv(csv_path, index=False)
        print(f"{name} split saved to {csv_path} ({len(split)} samples)")


In [11]:
print("Preparing attribution splits for ADA...")
prepare_ADA_splits()
print("ADA splits prepared successfully.")


Preparing attribution splits for ADA...
CodecFake: selected 50 samples (label=0)
ASVspoof2021: selected 50 samples (label=1)
FakeOrReal: selected 50 samples (label=2)
train split saved to /kaggle/working/csv/ADA_split/train.csv (90 samples)
val split saved to /kaggle/working/csv/ADA_split/val.csv (30 samples)
test split saved to /kaggle/working/csv/ADA_split/test.csv (30 samples)
ADA splits prepared successfully.


## ADA Module Implementation

In [12]:
import os
import torch
import random
import numpy as np
import pandas as pd
from torch import nn
from tqdm import tqdm
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt
import torch.utils.data as data
from sklearn.manifold import TSNE
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import label_binarize
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

class PathLabelDataset(Dataset):
    def __init__(self, csv_file):
        df = pd.read_csv(csv_file)
        self.samples = [(Path(p), l) for p, l in zip(df['path'], df['label'])]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        tensor = torch.load(Path(path).resolve()).float()
        return tensor, label

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim=256, num_heads=4):
        super().__init__()

        self.norm = nn.LayerNorm(embed_dim)

        self.attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True
        )

    def forward(self, z):

        # z comes as [batch, channels, time]
        z = z.permute(0, 2, 1)   # -> [batch, time, channels]

        z_norm = self.norm(z)

        attn_out, _ = self.attn(z_norm, z_norm, z_norm)

        z = z + attn_out  # residual connection

        z = z.permute(0, 2, 1)   # back to [batch, channels, time]

        return z

In [13]:
class AudioDeepfakeAttributionModel(nn.Module):
    def __init__(self, pretrained_autoencoder, num_classes=3):
        super().__init__()
        self.encoder = pretrained_autoencoder.encoder
        for name, param in self.encoder.named_parameters():
            if name not in ["10.weight", "10.bias"]:
                param.requires_grad = False

        # Multi Head attention model
        self.attention = MultiHeadSelfAttention(
            embed_dim=256, 
            num_heads=4)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        z = self.encoder(x)

        z = self.attention(z)   # MHSA + residual
        
        return self.classifier(z)

In [14]:
def train_model(model, train_loader, val_loader, device, epochs=20, lr=1e-4, save_path='/kaggle/working/models/ADA_model.pt'):
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best_val_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Train Loss: {total_loss / len(train_loader):.4f}")

        model.eval()
        val_loss = 0
        y_true, y_pred = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                output = model(x)
                val_loss += criterion(output, y).item()
                y_true.extend(y.cpu().numpy())
                y_pred.extend(output.argmax(dim=1).cpu().numpy())
        avg_val_loss = val_loss / len(val_loader)
        print(classification_report(y_true, y_pred, digits=4))

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            torch.save(model.state_dict(), save_path)
            print(f"Saved best model to {save_path}")

In [15]:
def evaluate_and_plot(model, loader, device, report_path, plot_dir):
    model.eval()
    y_true, y_pred = [], []
    logits_list = []
    latents = []

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Evaluating"):
            x, y = x.to(device), y.to(device)
            z = model.encoder(x)
            pooled = torch.nn.functional.adaptive_avg_pool1d(z, 1).squeeze(-1)
            latents.append(pooled.cpu())
            out = model(x)
            y_pred.extend(out.argmax(dim=1).cpu().numpy())
            y_true.extend(y.cpu().numpy())
            logits_list.append(out.cpu())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    probs = torch.softmax(torch.cat(logits_list), dim=1).numpy()
    latents = torch.cat(latents, dim=0).numpy()

    report = classification_report(y_true, y_pred, output_dict=True, digits=4)
    os.makedirs(os.path.dirname(report_path), exist_ok=True)
    pd.DataFrame(report).transpose().to_csv(report_path)

    matrix = confusion_matrix(y_true, y_pred)
    os.makedirs(plot_dir, exist_ok=True)
    plt.figure(figsize=(8, 6))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues')
    plt.title("Confusion Matrix")
    plt.savefig(os.path.join(plot_dir, "confusion_matrix.png"))
    plt.close()

    # ROC Curves
    y_bin = label_binarize(y_true, classes=list(range(3)))
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(3):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    plt.figure(figsize=(8, 6))
    for i in range(3):
        plt.plot(fpr[i], tpr[i], label=f"Class {i} AUC={roc_auc[i]:.2f}")
    plt.plot([0,1], [0,1], 'k--')
    plt.title("ROC Curves")
    plt.legend()
    plt.savefig(os.path.join(plot_dir, "roc_curves.png"))
    plt.close()

    # Precision-Recall Curves
    precision, recall, ap = {}, {}, {}
    for i in range(3):
        precision[i], recall[i], _ = precision_recall_curve(y_bin[:, i], probs[:, i])
        ap[i] = average_precision_score(y_bin[:, i], probs[:, i])

    plt.figure(figsize=(8, 6))
    for i in range(3):
        plt.plot(recall[i], precision[i], label=f"Class {i} AP={ap[i]:.2f}")
    plt.title("Precision-Recall Curves")
    plt.legend()
    plt.savefig(os.path.join(plot_dir, "pr_curves.png"))
    plt.close()

    # t-SNE 2D
    tsne_2d = TSNE(n_components=2, random_state=SEED, perplexity=1)
    emb_2d = tsne_2d.fit_transform(latents)
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=emb_2d[:,0], y=emb_2d[:,1], hue=y_true, palette="tab10", legend="full")
    plt.title("t-SNE (2D) of Latent Space")
    plt.savefig(os.path.join(plot_dir, "tsne_2d.png"))
    plt.close()

    # t-SNE 3D
    tsne_3d = TSNE(n_components=3, random_state=SEED, perplexity=1)
    emb_3d = tsne_3d.fit_transform(latents)
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    scatter = ax.scatter(emb_3d[:, 0], emb_3d[:, 1], emb_3d[:, 2], c=y_true, cmap="tab10")
    legend = ax.legend(*scatter.legend_elements(), title="Class")
    ax.add_artist(legend)
    ax.set_title("t-SNE (3D) of Latent Space")
    plt.savefig(os.path.join(plot_dir, "tsne_3d.png"))
    plt.close()

In [16]:
def compute_confidence_scores_with_preds(model, loader, device, output_csv="/kaggle/working/csv/confidence_scores.csv"):
    model.eval()
    scores, true_labels, pred_labels = [], [], []

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Computing confidence scores"):
            x = x.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            max_confidence, pred = probs.max(dim=1)
            
            scores.extend(max_confidence.cpu().numpy())
            pred_labels.extend(pred.cpu().numpy())
            true_labels.extend(y.numpy())

    df = pd.DataFrame({
        "confidence": scores,
        "true_label": true_labels,
        "pred_label": pred_labels
    })
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"Confidence scores with predictions saved to {output_csv}")

In [17]:
def find_confidence_thresholds(csv_path):
    import matplotlib.pyplot as plt

    df = pd.read_csv(csv_path)
    confidences = df["confidence"].values
    true_labels = df["true_label"].values
    pred_labels = df["pred_label"].values

    thresholds = np.linspace(0.0, 1.0, 500)
    best_cov = 0
    cov_thresh = 0

    cov_list = []

    for t in thresholds:
        mask = confidences >= t
        if np.sum(mask) == 0:
            continue

        preds = pred_labels[mask]
        targets = true_labels[mask]

        accuracy = np.mean(preds == targets)
        coverage = np.sum(mask) / len(true_labels)

        cov_list.append(coverage)

        if coverage >= 0.80 and accuracy > best_cov:
            best_cov = accuracy
            cov_thresh = t

    print(f"Coverage ≥80% Threshold: {cov_thresh:.4f} | Accuracy: {best_cov:.4f}")
    return cov_thresh

In [18]:
def predict_with_confidence_threshold(model, inputs, device, threshold):
    model.eval()
    inputs = inputs.to(device)
    
    with torch.no_grad():
        logits = model(inputs)
        probs = torch.softmax(logits, dim=1)
        max_confidence, predicted_class = torch.max(probs, dim=1)

        predictions = []
        confidences = max_confidence.cpu().numpy()

        for conf, pred in zip(confidences, predicted_class.cpu().numpy()):
            if conf >= threshold:
                predictions.append(pred)
            else:
                predictions.append(-1)  # -1 indicates uncertainty

    return predictions, confidences

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_csv = "/kaggle/working/csv/ADA_split"
model_path = "/kaggle/working/models/autoencoder.pt"
model_save_path = "/kaggle/working/models/ADA_model.pt"

autoencoder = DeepAutoencoder().to(device)
autoencoder.load_state_dict(torch.load(model_path, map_location=device))
model = AudioDeepfakeAttributionModel(autoencoder).to(device)

train_set = PathLabelDataset(os.path.join(base_csv, "train.csv"))
val_set = PathLabelDataset(os.path.join(base_csv, "val.csv"))
test_set = PathLabelDataset(os.path.join(base_csv, "test.csv"))

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader = DataLoader(val_set, batch_size=16)
test_loader = DataLoader(test_set, batch_size=16)

train_model(model, train_loader, val_loader, device, epochs=50, lr=1e-4, save_path=model_save_path)
model.load_state_dict(torch.load(model_save_path, map_location=device))

compute_confidence_scores_with_preds(model, train_loader, device,
                                     output_csv="/kaggle/working/csv/ADA/confidence_scores.csv")

find_confidence_thresholds("/kaggle/working/csv/ADA/confidence_scores.csv")

Epoch 1/50: 100%|██████████| 6/6 [00:00<00:00, 10.86it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(

Train Loss: 1.1005
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000        10
           1     0.0000    0.0000    0.0000        10
           2     0.3333    1.0000    0.5000        10

    accuracy                         0.3333        30
   macro avg     0.1111    0.3333    0.1667        30
weighted avg     0.1111    0.3333    0.1667        30

Saved best model to /kaggle/working/models/ADA_model.pt


Epoch 2/50: 100%|██████████| 6/6 [00:00<00:00, 38.48it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(

Train Loss: 1.1000
              precision    recall  f1-score   support

           0     0.3448    1.0000    0.5128        10
           1     0.0000    0.0000    0.0000        10
           2     1.0000    0.1000    0.1818        10

    accuracy                         0.3667        30
   macro avg     0.4483    0.3667    0.2315        30
weighted avg     0.4483    0.3667    0.2315        30



Epoch 3/50: 100%|██████████| 6/6 [00:00<00:00, 39.16it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(

...

Epoch 50/50: 100%|██████████| 6/6 [00:00<00:00, 39.68it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len

Train Loss: 1.0988
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000        10
           1     0.3333    1.0000    0.5000        10
           2     0.0000    0.0000    0.0000        10

    accuracy                         0.3333        30
   macro avg     0.1111    0.3333    0.1667        30
weighted avg     0.1111    0.3333    0.1667        30



Computing confidence scores: 100%|██████████| 6/6 [00:00<00:00, 37.82it/s]

Confidence scores with predictions saved to /kaggle/working/csv/ADA/confidence_scores.csv
Coverage ≥80% Threshold: 0.0000 | Accuracy: 0.3333


np.float64(0.0)

In [20]:
evaluate_and_plot(
        model=model,
        loader=test_loader,
        device=torch.device("cuda"), 
        report_path="/kaggle/working/reports/ADA_test_report.csv",
        plot_dir="/kaggle/working/plots/ADA/"
    )

Evaluating: 100%|██████████| 2/2 [00:00<00:00, 32.09it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(

# Audio Model Deepfake Recognition

## Prepare ADMR Data Splits

In [21]:
def prepare_ADMR_splits():
    # Output
    output_csv_dir = "/kaggle/working/csv/ADMR_split"
    os.makedirs(output_csv_dir, exist_ok=True)

    dataset = CodecFakeMultiClassDataset(
        root_dir="/kaggle/working/audio/CodecFake/fake",
        seed=SEED
    )

    # Saving the dataset samples
    all_samples = dataset.samples

    # Splitting the dataset into train, val, and test sets
    train_val, test = train_test_split(all_samples, test_size=0.2, stratify=[lbl for _, lbl in all_samples], random_state=SEED)
    train, val = train_test_split(train_val, test_size=0.25, stratify=[lbl for _, lbl in train_val], random_state=SEED)
    # Result: 60% train, 20% val, 20% test

    splits = {
        "train": train,
        "val": val,
        "test": test
    }

    # Save splits to CSV files
    for name, split in splits.items():
        df = pd.DataFrame([(str(p), l - 1) for p, l in split], columns=["path", "label"])
        save_path = os.path.join(output_csv_dir, f"{name}.csv")
        df.to_csv(save_path, index=False)
        print(f"Saved {name} split with {len(split)} samples.")


In [22]:
print("\nPreparing attribution splits for ADMR...")
prepare_ADMR_splits()
print("ADMR splits prepared successfully.")


Preparing attribution splits for ADMR...
Class 1: 9 samples
Class 2: 9 samples
Class 3: 8 samples
Class 4: 8 samples
Class 5: 8 samples
Class 6: 8 samples
Total samples: 50
Saved train split with 30 samples.
Saved val split with 10 samples.
Saved test split with 10 samples.
ADMR splits prepared successfully.


## ADMR Module Implementation

In [23]:
import os
import torch
import random
import numpy as np
import pandas as pd
from torch import nn
from tqdm import tqdm
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt
import torch.utils.data as data
from sklearn.manifold import TSNE
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import label_binarize
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

class ADMR_model(nn.Module):
    def __init__(self, pretrained_autoencoder):
        super(ADMR_model, self).__init__()
        self.encoder = pretrained_autoencoder.encoder
        for name, param in self.encoder.named_parameters():
            if name not in ["10.weight", "10.bias"]:
                param.requires_grad = False


        self.attention = MultiHeadSelfAttention(
                embed_dim=256,
                num_heads=4
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 6)  # 6 classes
        )

    def forward(self, x):
        z = self.encoder(x)
        z = self.attention(z)
        out = self.classifier(z)
        return out
    
    def load_ADMR_model(autoencoder_path, ADMR_model_path, device):
        # Loads the pretrained autoencoder
        autoencoder = DeepAutoencoder().to(device)
        autoencoder.load_state_dict(torch.load(autoencoder_path, map_location=device))

        # Loads the pretrained ADMR model
        model = ADMR_model(pretrained_autoencoder=autoencoder).to(device)
        model.load_state_dict(torch.load(ADMR_model_path, map_location=device))
        model.eval()

        print(f"ADMR model loaded from {ADMR_model_path}")
        return model


In [24]:
def train_ADMR_model(model, train_loader, val_loader, device, epochs=10, lr=1e-4, save_path='/kaggle/working/models/ADMR_model.pt'):
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    best_val_loss = float("inf")

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0
        y_true, y_pred = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss = criterion(logits, y)
                val_loss += loss.item()
                y_true.extend(y.cpu().numpy())
                y_pred.extend(logits.argmax(dim=1).cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            if os.path.exists(save_path):
                os.remove(save_path)
            torch.save(model.state_dict(), save_path)
            print(f"Model saved to {save_path}")

        print(f"\nEpoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(classification_report(y_true, y_pred, digits=4))

In [25]:
def evaluate_model(model, test_loader, device, report_path="/kaggle/working/reports/ADMR/test_report.csv", plot_path="/kaggle/working/plots/ADMR/"):
    model.eval()
    y_true, y_pred = [], []
    logits_list = []
    latent_features = []

    with torch.no_grad():
        for x, y in tqdm(test_loader, desc="Evaluating on test set"):
            x, y = x.to(device), y.to(device)
            z = model.encoder(x)  # shape: [B, 256, T]
            pooled = torch.nn.functional.adaptive_avg_pool1d(z, 1).squeeze(-1)  # shape: [B, 256]
            latent_features.append(pooled.cpu())

            logits = model(x)
            preds = logits.argmax(dim=1)

            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            logits_list.append(logits.cpu())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    logits_all = torch.cat(logits_list, dim=0)
    probs = torch.softmax(logits_all, dim=1).numpy()
    latents = torch.cat(latent_features, dim=0).numpy()

    # Compute metrics
    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, output_dict=True, digits=4)
    matrix = confusion_matrix(y_true, y_pred)

    # Print and save classification report
    print(f"\nTest Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=4))

    os.makedirs(os.path.dirname(report_path), exist_ok=True)
    pd.DataFrame(report).transpose().to_csv(report_path)
    print(f"Classification report saved to {report_path}")

    # Confusion matrix
    cm_path = os.path.join(plot_path, "confusion_matrix.png")
    os.makedirs(os.path.dirname(cm_path), exist_ok=True)
    plt.figure(figsize=(8, 6))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=[f"C{i+1}" for i in range(6)], yticklabels=[f"C{i+1}" for i in range(6)])
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Confusion Matrix")
    plt.savefig(cm_path)
    plt.close()
    print(f"Confusion matrix saved to {cm_path}")

    # --- t-SNE 2D ---
    tsne_2d_path = os.path.join(plot_path, "tsne_2d.png")
    tsne_2d = TSNE(n_components=2, random_state=SEED, perplexity=1)
    emb_2d = tsne_2d.fit_transform(latents)
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=emb_2d[:, 0], y=emb_2d[:, 1], hue=y_true, palette="tab10", legend="full")
    plt.title("t-SNE (2D) of Test Latent Representations")
    plt.savefig(tsne_2d_path)
    plt.close()
    print(f"t-SNE 2D plot saved to {tsne_2d_path}")

    # --- t-SNE 3D ---
    tsne_3d_path = os.path.join(plot_path, "tsne_3d.png")
    tsne_3d = TSNE(n_components=3, random_state=SEED, perplexity=1)
    emb_3d = tsne_3d.fit_transform(latents)
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    scatter = ax.scatter(emb_3d[:, 0], emb_3d[:, 1], emb_3d[:, 2], c=y_true, cmap="tab10")
    legend = ax.legend(*scatter.legend_elements(), title="Class")
    ax.add_artist(legend)
    ax.set_title("t-SNE (3D) of Test Latent Representations")
    plt.savefig(tsne_3d_path)
    plt.close()
    print(f"t-SNE 3D plot saved to {tsne_3d_path}")

    # --- ROC & PR Curves ---
    y_bin = label_binarize(y_true, classes=list(range(6)))
    fpr, tpr, roc_auc = dict(), dict(), dict()
    precision, recall, ap = dict(), dict(), dict()

    for i in range(6):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], probs[:, i])
        precision[i], recall[i], _ = precision_recall_curve(y_bin[:, i], probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        ap[i] = average_precision_score(y_bin[:, i], probs[:, i])

    # ROC Curve
    roc_path = "/kaggle/working/plots/ADMR/roc_curves.png"
    plt.figure(figsize=(8, 6))
    for i in range(6):
        plt.plot(fpr[i], tpr[i], label=f"C{i+1} (AUC={roc_auc[i]:.2f})")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves")
    plt.legend()
    plt.savefig(roc_path)
    plt.close()
    print(f"ROC curves saved to {roc_path}")

    # PR Curve
    pr_path = "/kaggle/working/plots/ADMR/pr_curves.png"
    plt.figure(figsize=(8, 6))
    for i in range(6):
        plt.plot(recall[i], precision[i], label=f"C{i+1} (AP={ap[i]:.2f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curves")
    plt.legend()
    plt.savefig(pr_path)
    plt.close()
    print(f"Precision-Recall curves saved to {pr_path}")

In [26]:
def compute_confidence_scores_with_preds(model, loader, device, output_csv="/kaggle/working/csv/ADMR/confidence_scores.csv"):
    model.eval()
    scores, true_labels, pred_labels = [], [], []

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Computing confidence scores"):
            x = x.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            max_confidence, pred = probs.max(dim=1)
            
            scores.extend(max_confidence.cpu().numpy())
            pred_labels.extend(pred.cpu().numpy())
            true_labels.extend(y.numpy())

    df = pd.DataFrame({
        "confidence": scores,
        "true_label": true_labels,
        "pred_label": pred_labels
    })
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"Confidence scores with predictions saved to {output_csv}")

In [27]:
def find_confidence_thresholds(csv_path):
    import matplotlib.pyplot as plt

    df = pd.read_csv(csv_path)
    confidences = df["confidence"].values
    true_labels = df["true_label"].values
    pred_labels = df["pred_label"].values

    thresholds = np.linspace(0.0, 1.0, 500)
    best_cov = 0
    cov_thresh = 0

    cov_list = []

    for t in thresholds:
        mask = confidences >= t
        if np.sum(mask) == 0:
            continue

        preds = pred_labels[mask]
        targets = true_labels[mask]

        accuracy = np.mean(preds == targets)
        coverage = np.sum(mask) / len(true_labels)

        cov_list.append(coverage)

        if coverage >= 0.80 and accuracy > best_cov:
            best_cov = accuracy
            cov_thresh = t

    print(f"Coverage ≥80% Threshold: {cov_thresh:.4f} | Accuracy: {best_cov:.4f}")
    return cov_thresh

In [28]:
def predict_with_confidence_threshold(model, inputs, device, threshold=0.95): #0.8
    model.eval()
    inputs = inputs.to(device)
    
    with torch.no_grad():
        logits = model(inputs)
        probs = torch.softmax(logits, dim=1)
        max_confidence, predicted_class = torch.max(probs, dim=1)

        predictions = []
        confidences = max_confidence.cpu().numpy()

        for conf, pred in zip(confidences, predicted_class.cpu().numpy()):
            if conf >= threshold:
                predictions.append(pred)
            else:
                predictions.append(-1)  # -1 indicates uncertainty

    return predictions, confidences

In [29]:
model_path = "/kaggle/working/models/autoencoder.pt"
csv_path = "/kaggle/working/csv/ADMR_split"
model_save_path = "/kaggle/working/models/ADMR_model.pt"
generator = torch.Generator().manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
autoencoder = DeepAutoencoder().to(device)
autoencoder.load_state_dict(torch.load(model_path, map_location=device))

model = ADMR_model(pretrained_autoencoder=autoencoder).to(device)
    
train_set = PathLabelDataset(os.path.join(csv_path, "train.csv"))
val_set = PathLabelDataset(os.path.join(csv_path, "val.csv"))
test_set = PathLabelDataset(os.path.join(csv_path, "test.csv"))

print(f"Train set size: {len(train_set)}\nValidation set size: {len(val_set)}\nTest set size: {len(test_set)}")

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader = DataLoader(val_set, batch_size=16)

train_ADMR_model(model, train_loader, val_loader, device, epochs=50, lr=1e-4, save_path=model_save_path)
print(f"ADMR model trained and saved to {model_save_path}")


model.load_state_dict(torch.load(model_save_path, map_location=device))

test_loader = DataLoader(test_set, batch_size=16)
evaluate_model(model, test_loader, device, report_path="/kaggle/working/reports/ADMR/test_report.csv", plot_path="/kaggle/working/plots/ADMR/")
print("Model evaluation completed.")

compute_confidence_scores_with_preds(model, train_loader, device)
find_confidence_thresholds("/kaggle/working/csv/ADMR/confidence_scores.csv")

Train set size: 30
Validation set size: 10
Test set size: 10


Epoch 1/50: 100%|██████████| 2/2 [00:00<00:00, 27.80it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(

Model saved to /kaggle/working/models/ADMR_model.pt

Epoch 1 | Train Loss: 1.7939 | Val Loss: 1.7908
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000         2
           1     0.2000    1.0000    0.3333         2
           2     0.0000    0.0000    0.0000         1
           3     0.0000    0.0000    0.0000         2
           4     0.0000    0.0000    0.0000         1
           5     0.0000    0.0000    0.0000         2

    accuracy                         0.2000        10
   macro avg     0.0333    0.1667    0.0556        10
weighted avg     0.0400    0.2000    0.0667        10



Epoch 2/50: 100%|██████████| 2/2 [00:00<00:00, 28.21it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(

Model saved to /kaggle/working/models/ADMR_model.pt

Epoch 2 | Train Loss: 1.7930 | Val Loss: 1.7904
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000         2
           1     0.2000    1.0000    0.3333         2
           2     0.0000    0.0000    0.0000         1
           3     0.0000    0.0000    0.0000         2
           4     0.0000    0.0000    0.0000         1
           5     0.0000    0.0000    0.0000         2

    accuracy                         0.2000        10
   macro avg     0.0333    0.1667    0.0556        10
weighted avg     0.0400    0.2000    0.0667        10



...

Evaluating on test set: 100%|██████████| 1/1 [00:00<00:00, 50.47it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize(


Test Accuracy: 0.2000

Classification Report:
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000         2
           1     0.2000    1.0000    0.3333         2
           2     0.0000    0.0000    0.0000         2
           3     0.0000    0.0000    0.0000         1
           4     0.0000    0.0000    0.0000         2
           5     0.0000    0.0000    0.0000         1

    accuracy                         0.2000        10
   macro avg     0.0333    0.1667    0.0556        10
weighted avg     0.0400    0.2000    0.0667        10

Classification report saved to /kaggle/working/reports/ADMR/test_report.csv
Confusion matrix saved to /kaggle/working/plots/ADMR/confusion_matrix.png
t-SNE 2D plot saved to /kaggle/working/plots/ADMR/tsne_2d.png
t-SNE 3D plot saved to /kaggle/working/plots/ADMR/tsne_3d.png
ROC curves saved to /kaggle/working/plots/ADMR/roc_curves.png
Precision-Recall curves saved to /kaggle/working/plots/ADMR/pr_curves.png


Computing confidence scores: 100%|██████████| 2/2 [00:00<00:00, 46.61it/s]

Confidence scores with predictions saved to /kaggle/working/csv/ADMR/confidence_scores.csv
Coverage ≥80% Threshold: 0.0000 | Accuracy: 0.1667


np.float64(0.0)